<a href="https://colab.research.google.com/github/ravivarma1701/sample_docs/blob/main/langchain_docling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-qdrant langchain-groq
!pip install -q sentence-transformers qdrant-client langchain-docling

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [ ]:

from langchain_docling.loader import DoclingLoader
from langchain_docling.loader import ExportType
from docling.chunking import HierarchicalChunker
from langchain_docling.loader import ExportType

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EXPORT_TYPE = ExportType.DOC_CHUNKS
FILE_PATH = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf"
chunker = HierarchicalChunker(tokenizer=EMBED_MODEL_ID)

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=chunker
    )

docs = loader.load()
print(docs)

In [ ]:
def convert_chunk(doc_chunk) -> dict:
    """
    Convert a Docling DocChunk into a plain dict.

    headings   → list preserved as-is
    content    → paragraph text
    chunk_text → breadcrumb + content  (what gets embedded)
    """
    print(f'===={doc_chunk.metadata}')
    headings = (
    doc_chunk.metadata
    .get("dl_meta", {})
    .get("headings", [])
)  # LangChain stores metadata here
    content = doc_chunk.page_content.strip()
    breadcrumb = " > ".join(headings)
    chunk_text = f"{breadcrumb}\n\n{content}" if breadcrumb else content
    return {
        "headings": headings,
        "content": content,
        "chunk_text": chunk_text
    }


chunks = [convert_chunk(c) for c in docs]


In [ ]:
for chunk in chunks[:3]:
    print("─" * 60)
    print(f"headings   : {chunk['headings']}")
    print(f"content    : {chunk['content'][:200]}…")

In [ ]:

from langchain.text_splitter import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(
    headers=["##", "###"],   # split by header levels
    chunk_size=800,
    chunk_overlap=100
)

doc_chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(doc_chunks)}")

# Inspect a sample
print(doc_chunks[0].page_content[:200], "…")